# 🧠 Model Pipeline – Wide Dataset (Cross-Subject)
**File:** `normal_task_features_wide.csv`  
**Task:** 9-class EEG Scenario Classification  
**Protocol:** Cross-subject evaluation using **GroupKFold** by `subject_id`  
**Output folder:** `wide_model_outputs/`

> **Cross-subject protocol:** Models are trained on data from one group of subjects and tested on an entirely different group of subjects. This is the rigorous, real-world BCI evaluation protocol.

### Models
1. Random Forest · 2. Logistic Regression · 3. KNN · 4. XGBoost · 5. LightGBM · 6. AdaBoost  
7. **Stacking Ensemble** (RF + XGB + LGBM → Logistic Regression meta-learner)  
8. **Boosting Ensemble** (GradientBoosting)


## 0. Setup

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (GroupKFold, GroupShuffleSplit,
                                     cross_validate, StratifiedKFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, roc_auc_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

OUTPUT_DIR = "wide_model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
SCENARIO_LABELS = {
    0:"Lift L.Hand", 1:"Lift R.Hand", 2:"Lift L.Leg",  3:"Lift R.Leg",
    4:"Open Mouth",  5:"Nod Head",    6:"Shake Head",   7:"Want Water", 8:"Use Bathroom"
}
LABELS_S = [f"S{i+1}" for i in range(9)]
CMAPS    = ['Blues','Greens','Oranges','Reds','Purples','YlOrBr','GnBu','RdPu','BuPu']
print("Setup complete.")


## 1. Load & Preprocess

In [ ]:
PATH = "normal_task_features_wide.csv"
df = pd.read_csv(PATH)
print(f"Shape: {df.shape}")
print(f"Subjects: {df['subject_id'].nunique()}, Scenarios: {df['scenario_num'].nunique()}")
print(f"Rows per subject: {df.groupby('subject_id').size().describe().to_dict()}")


In [ ]:
feature_cols = [c for c in df.columns
                if c not in ['subject_id', 'scenario', 'scenario_num']]

# Feature matrix – keep 0 values (informative per domain knowledge)
# Only impute actual NaN with median
X_raw = df[feature_cols].values.astype(float)
y     = df['scenario_num'].values - 1          # 0-indexed (required by XGBoost)
groups = LabelEncoder().fit_transform(df['subject_id'])  # group labels for GroupKFold

imputer = SimpleImputer(strategy='median')      # NaN → median; zeros untouched
X = imputer.fit_transform(X_raw)

print(f"Feature matrix: {X.shape}")
print(f"Zero count (preserved): {(X == 0).sum()}")
print(f"NaN after imputation  : {np.isnan(X).sum()}")
print(f"Classes               : {np.unique(y)}")
print(f"Class distribution    :\n{pd.Series(y).value_counts().sort_index().to_string()}")


## 2. Train / Test Split (Group-based)

In [ ]:
# GroupShuffleSplit ensures no subject leaks between train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train : {X_train.shape}  |  subjects: {len(np.unique(groups[train_idx]))}")
print(f"Test  : {X_test.shape}   |  subjects: {len(np.unique(groups[test_idx]))}")
print("✅ No subject overlap between train and test (cross-subject protocol)")


## 3. Define Model Pipelines

In [ ]:
# ── Base Models ────────────────────────────────────────────────────────────────
models = {
    "Random Forest": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=15,
                                       min_samples_leaf=2, random_state=42, n_jobs=-1))
    ]),
    "Logistic Regression": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs',
                                   random_state=42))
    ]),
    "KNN": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7, metric='euclidean', n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric='mlogloss', random_state=42,
                              n_jobs=-1, tree_method='hist'))
    ]),
    "LightGBM": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                               subsample=0.8, colsample_bytree=0.8,
                               random_state=42, n_jobs=-1, verbose=-1))
    ]),
    "AdaBoost": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', AdaBoostClassifier(n_estimators=150, learning_rate=0.5,
                                   random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                           learning_rate=0.1, subsample=0.8,
                                           random_state=42))
    ]),
}

# ── Stacking Ensemble ──────────────────────────────────────────────────────────
stk_estimators = [
    ('rf',   RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)),
    ('xgb',  XGBClassifier(n_estimators=150, tree_method='hist',
                            eval_metric='mlogloss', random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=150, random_state=42, n_jobs=-1, verbose=-1)),
    ('knn',  KNeighborsClassifier(n_neighbors=7)),
]
models["Stacking (RF+XGB+LGBM+KNN)"] = Pipeline([
    ('sc',  StandardScaler()),
    ('clf', StackingClassifier(
        estimators=stk_estimators,
        final_estimator=LogisticRegression(max_iter=2000, random_state=42),
        cv=5, passthrough=False, n_jobs=-1
    ))
])
print(f"Models defined: {list(models.keys())}")


## 4. Group K-Fold Cross-Validation

In [ ]:
N_SPLITS = 10   # ~17 subjects held-out per fold
gkf = GroupKFold(n_splits=N_SPLITS)
cv_results = {}

print(f"Running {N_SPLITS}-Fold Group Cross-Validation...")
print("(Stacking skipped in CV – evaluated on test set only)\n")

for name, pipe in models.items():
    if "Stacking" in name:
        print(f"  {name:35s}: [skipped in CV]")
        continue
    res = cross_validate(pipe, X, y, cv=gkf, groups=groups,
                         scoring='accuracy', n_jobs=-1, return_train_score=True)
    cv_results[name] = res
    te = res['test_score'];  tr = res['train_score']
    print(f"  {name:35s}: test={te.mean():.4f}±{te.std():.4f}  train={tr.mean():.4f}")


## 5. Train All Models & Evaluate on Test Set

In [ ]:
test_results = {}

for name, pipe in models.items():
    print(f"  Training {name} ...", end='  ', flush=True)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    test_results[name] = {'accuracy': acc, 'f1_weighted': f1, 'y_pred': y_pred}
    print(f"Acc={acc:.4f}  F1={f1:.4f}")


## 6. Visualisations

In [ ]:
# ── FIG 1 – GroupKFold CV Bar Chart ───────────────────────────────────────────
cv_names = list(cv_results.keys())
cv_means = [cv_results[n]['test_score'].mean() for n in cv_names]
cv_stds  = [cv_results[n]['test_score'].std()  for n in cv_names]
cv_tr    = [cv_results[n]['train_score'].mean() for n in cv_names]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(cv_names)); w = 0.35
b1 = ax.bar(x-w/2, cv_means, w, yerr=cv_stds, label='Test (cross-subject)',
            color='steelblue', alpha=0.85, capsize=4)
b2 = ax.bar(x+w/2, cv_tr,    w, label='Train', color='darkorange', alpha=0.7)
for b, v in zip(b1, cv_means):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.015, f'{v:.3f}',
            ha='center', fontsize=8)
ax.axhline(1/9, color='red', ls='--', alpha=0.5, label='Chance (11.1%)')
ax.set_xticks(x); ax.set_xticklabels(cv_names, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.2)
ax.set_title(f"{N_SPLITS}-Fold Group CV – Wide Dataset (Cross-Subject)",
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_cv_group_kfold.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 2 – Test Performance ───────────────────────────────────────────────────
nt = list(test_results.keys())
at = [test_results[n]['accuracy']    for n in nt]
ft = [test_results[n]['f1_weighted'] for n in nt]
x  = np.arange(len(nt)); w = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x-w/2, at, w, label='Accuracy',    color='steelblue',  alpha=0.85)
b2 = ax.bar(x+w/2, ft, w, label='F1-Weighted', color='darkorange', alpha=0.85)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=8)
ax.axhline(1/9, color='red', ls='--', alpha=0.4, label='Chance (11.1%)')
ax.set_xticks(x); ax.set_xticklabels(nt, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.15)
ax.set_title("Test Set Performance – Wide Dataset (Cross-Subject)",
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_test_performance.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 3 – Best Model Confusion Matrix ───────────────────────────────────────
best_name = max(test_results, key=lambda n: test_results[n]['accuracy'])
print(f"Best model: {best_name}  (Acc={test_results[best_name]['accuracy']:.4f})")

cm = confusion_matrix(y_test, test_results[best_name]['y_pred'])
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=LABELS_S, yticklabels=LABELS_S, linewidths=0.5)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix – {best_name}\n(Wide, Cross-Subject Test)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_confusion_matrix_best.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 4 – Classification Report Table ───────────────────────────────────────
rpt = classification_report(y_test, test_results[best_name]['y_pred'], output_dict=True)
rdf = pd.DataFrame(rpt).T

fig, ax = plt.subplots(figsize=(13, 4)); ax.axis('off')
rows = [[f"S{i+1} – {SCENARIO_LABELS[i]}",
         f"{rdf.loc[str(i),'precision']:.3f}",
         f"{rdf.loc[str(i),'recall']:.3f}",
         f"{rdf.loc[str(i),'f1-score']:.3f}",
         f"{int(rdf.loc[str(i),'support'])}"]
        for i in range(9) if str(i) in rdf.index]
t = ax.table(cellText=rows,
             colLabels=['Scenario','Precision','Recall','F1-Score','Support'],
             cellLoc='center', loc='center', colColours=['#2E75B6']*5)
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.2, 1.7)
for j in range(5): t[0,j].set_text_props(color='white', fontweight='bold')
ax.set_title(f"Classification Report – {best_name} (Wide)",
             fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_classification_report.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 5 – RF Feature Importance ─────────────────────────────────────────────
rf_imp = models["Random Forest"].named_steps['clf'].feature_importances_
fi = (pd.DataFrame({'feature': feature_cols, 'importance': rf_imp})
      .sort_values('importance', ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi['feature'], fi['importance'],
        color=plt.cm.YlOrRd(np.linspace(0.4, 0.9, 20))[::-1])
ax.set_xlabel("Importance")
ax.set_title("Top 20 Feature Importances – Random Forest (Wide)",
             fontsize=12, fontweight='bold')
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_rf_feature_importance.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 6 – XGBoost Feature Importance ────────────────────────────────────────
xgb_imp = models["XGBoost"].named_steps['clf'].feature_importances_
fi_xgb = (pd.DataFrame({'feature': feature_cols, 'importance': xgb_imp})
           .sort_values('importance', ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi_xgb['feature'], fi_xgb['importance'],
        color=plt.cm.BuGn(np.linspace(0.4, 0.9, 20))[::-1])
ax.set_xlabel("Importance")
ax.set_title("Top 20 Feature Importances – XGBoost (Wide)",
             fontsize=12, fontweight='bold')
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_xgb_feature_importance.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 7 – Stacking Confusion Matrix ─────────────────────────────────────────
stack_key = "Stacking (RF+XGB+LGBM+KNN)"
cm_s = confusion_matrix(y_test, test_results[stack_key]['y_pred'])
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=LABELS_S, yticklabels=LABELS_S, linewidths=0.5)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix – Stacking Ensemble (Wide)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_stacking_confusion_matrix.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 8 – All Confusion Matrices Grid ───────────────────────────────────────
nm = len(test_results); ncols = 4; nrows = (nm + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows))
fig.suptitle("All Confusion Matrices – Wide Dataset (Cross-Subject)",
             fontsize=13, fontweight='bold')

for idx, (name, res) in enumerate(test_results.items()):
    ax  = axes.flat[idx]
    cmi = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cmi, annot=True, fmt='d', cmap=CMAPS[idx % len(CMAPS)],
                ax=ax, xticklabels=LABELS_S, yticklabels=LABELS_S,
                linewidths=0.3, cbar=False)
    ax.set_title(f"{name}\nAcc={res['accuracy']:.3f}", fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=6)

for idx in range(nm, nrows*ncols): axes.flat[idx].axis('off')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/08_all_confusion_matrices.png", bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 9 – Overfitting Analysis ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
names_ov  = list(cv_results.keys())
train_acc = [cv_results[n]['train_score'].mean() for n in names_ov]
test_acc  = [cv_results[n]['test_score'].mean()  for n in names_ov]
x = np.arange(len(names_ov)); w = 0.35

ax.bar(x-w/2, train_acc, w, label='Train Acc', color='steelblue', alpha=0.85)
ax.bar(x+w/2, test_acc,  w, label='Test Acc (cross-subj)', color='salmon', alpha=0.85)
for i, (tr, te) in enumerate(zip(train_acc, test_acc)):
    ax.text(i, max(tr, te) + 0.02, f'Δ={tr-te:.2f}', ha='center',
            fontsize=8, color='darkred', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(names_ov, rotation=25, ha='right')
ax.axhline(1/9, color='red', ls='--', alpha=0.3, label='Chance')
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.2)
ax.set_title("Train vs Test – Overfitting Analysis (Wide, Cross-Subject)",
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/09_overfitting_analysis.png", bbox_inches='tight')
plt.show()


## 7. Save Results

In [ ]:
summary = pd.DataFrame([{
    'Model':            n,
    'Test_Accuracy':    r['accuracy'],
    'F1_Weighted':      r['f1_weighted'],
    'GroupKFold_CV_Mean': cv_results[n]['test_score'].mean() if n in cv_results else None,
    'GroupKFold_CV_Std':  cv_results[n]['test_score'].std()  if n in cv_results else None,
} for n, r in test_results.items()]).sort_values('Test_Accuracy', ascending=False)

summary.to_csv(f"{OUTPUT_DIR}/model_results_summary.csv", index=False)
print(summary.to_string(index=False))
print(f"\n✅ Results saved to {OUTPUT_DIR}/model_results_summary.csv")
